# Memory Management

## 🟢 Basic Unit of Measurement
- pico (p) = 10^-12
- nano (n) = 10^-9
- micro (µ) = 10^-6
- milli (m) = 10^-3
- base unit = 10^0
- kilo (k) = 10^3
- mega (M) = 10^6
- giga (G) = 10^9
- tera (T) = 10^12

__
- kibi (Ki) = 2^10 = 1,024
- mebi (Mi) = 2^20 = 1,048,576
- gibi (Gi) = 2^30 = 1,073,741,824
- tebi (Ti) = 2^40 = 1,099,511,627,776

---

## 🟢 Storage Types
### Disk storage
- Physical, long-term storage such as: HDD, SSD
- Holds files, programs, OS data
- Non-volatile: data stays after power off

### Virtual storage (usually virtual memory)
- A memory management technique used by the OS
- Makes processes think they have a large continuous memory space
- Uses RAM plus part of disk called swap/page file
- Slower than RAM because disk is much slower
- Lets more programs run than fit entirely in physical RAM

### Key difference
- Disk storage stores files permanently
- Virtual storage/memory is an OS abstraction for running programs, sometimes backed by disk

## 🟢 Memory Size & Bits
**address bits = log2(N)**

### Examples
- 1 KB = 2^10 bytes → 10 bits
- 4 KB = 2^12 bytes → 12 bits
- 1 MB = 2^20 bytes → 20 bits
- 1 GB = 2^30 bytes → 30 bits
- 4 GB = 2^32 bytes → 32 bits
- 16 GB = 2^34 bytes → 34 bits
- 1 TB = 2^40 bytes → 40 bits

In [2]:
import math
N = 16000000000
bits_needed = math.ceil(math.log2(N))
print(bits_needed)

34


---

## 🟢 Hierarchical Storage Organization (Hardware)
Computer systems organize storage as a hierarchy because no single technology can optimize speed, capacity, and cost at the same time.

## Hierarchy High to Low
- Registers
    - Smallest and fastest storage
    - Located inside the CPU
    - Hold operands, addresses, and intermediate results for current instructions
- Cache (L1, L2, L3)
    - Stores recently or frequently used data and instructions
    - Reduces average memory access time
    - L1 is smallest and fastest; L3 is larger and slower
- Primary storage
    - Usually main memory (RAM)
    - Holds currently running programs and active data
    - CPU does not execute directly from disk; code and data must be loaded into RAM first
- Secondary storage
    - SSDs or HDDs
    - Much larger, much cheaper per bit, but much slower
    - Used for files, installed programs, and long-term persistence

As you move up the hierarchy:
- Access time decreases → faster access
- Access speed increases
- Cost per bit increases
- Capacity decreases

As you move down the hierarchy:
- storage gets larger and cheaper
- but also slower

The hierarchy works because programs tend to show locality of reference:
- Temporal locality: recently used data is likely to be used again soon
- Spatial locality: data near recently used addresses is likely to be used soon

That is why keeping a small amount of data in fast storage can dramatically improve performance.

---

## 🟢 Transfers Between Storage Levels
Data moves between adjacent levels of the hierarchy so the CPU can access it efficiently.

1. Cache ↔ Primary storage
- Managed mostly by hardware
- Happens automatically on cache hits and cache misses
- Transfers occur in fixed-size blocks called cache lines
- Designed to be extremely fast because the CPU depends on it every cycle
- Software usually does not explicitly decide each transfer
- **Cache hit:** requested data is already in cache
- **Cache miss:** data must be fetched from RAM
- Performance depends heavily on keeping the hit rate high

2. Primary storage ↔ Secondary storage
- Managed largely by the operating system
- Used when RAM is not enough for all active processes
- Transfers happen in units such as pages
- This is the basis of virtual memory
- If a needed page is not in RAM, a page fault occurs and the OS loads it from disk
- This is much slower than cache or RAM access

### Page
- Fixed-size block of virtual memory, often 4 KiB
- OS moves pages between disk and RAM

### Swap
- Disk area used as backing storage for pages that do not currently fit in RAM

### Huge Penalty for Bad Decision
A poor replacement choice can cause:
- repeated page faults
- excessive disk I/O
- **thrashing**, where the system spends more time moving pages than running programs

### Software vs Hardware Control Summary
- Cache management is hardware-driven because decisions must be made in nanoseconds
- Paging/swapping is OS-driven because it involves process state, protection, disk I/O, and broader system policy

---

## 🟢 Storage Management
Storage management is about how the OS moves programs and data between:
- primary storage = main memory (RAM)
- secondary storage = disk/SSD

The OS must decide:
- when to bring data into memory
- where to put it
- what to remove when memory is full

Those are the core transfer strategies:
- fetch
- placement
- replacement

---

## 🟢 Fetch (Placement) Strategies
When should a program page/segment/block be brought from secondary storage into main memory?

### 1. Demand fetch
Bring data into memory only when it is actually needed. This is the basis of demand paging.

**Example:**
- A program starts
- It tries to access a page not currently in RAM
- A page fault occurs
- The OS loads that page from disk into RAM

**Advantages**
- avoids loading unused code/data
- saves memory
- supports larger virtual address spaces
- good for programs with locality

**Disadvantages**
- first access is slow
- too many faults can cause heavy overhead
- may lead to thrashing

Demand fetch = reactive

---

### 2. Anticipatory fetch
Bring data into memory before it is explicitly requested, based on prediction. Also called prefetching.

**Example:**
- if page 10 is used, the OS may also load page 11
- if sequential file access is detected, the system reads ahead

**Advantages**
- can reduce waiting time
- useful when access pattern is predictable
- improves performance for sequential or regular workloads

**Disadvantages**
- wrong prediction wastes memory and I/O
- may evict useful pages
- adds policy complexity

Anticipatory fetch = proactive

---

## 🟢 Placement strategies
Placement strategy answers: Where in main memory should the incoming program/page/segment be placed? This matters most in schemes where memory is allocated in variable-size chunks, such as:
- contiguous allocation
- partitioned allocation
- segmentation

### 1. First fit
Place the item in the first free block large enough.
- simple
- fast
- commonly used

### 2. Best fit
Place the item in the smallest free block that is still large enough.
- tries to reduce wasted space
- may leave many tiny unusable gaps

### 3. Worst fit
Place the item in the largest free block.
- tries to leave sizable remaining holes
- usually not as effective in practice

### In paging
Placement is simpler:
- any free page frame can hold any page
- physical placement matters less because address translation maps virtual to physical

---

## 🟢  Replacement strategies
Replacement strategy answers: If memory is full, what should be removed to make room?
This is especially important in **paging** and **segmentation with paging**. Goal of replacement is to minimize:
- page faults
- disk traffic
- process delay

### 1. FIFO (First-In, First-Out)
Replace the oldest page in memory.
- simple
- may remove a heavily used page
- can suffer from Belady’s anomaly

### 2. Optimal (MIN)
Replace the page that will not be used for the longest future time.
- best possible in theory
- impossible to implement exactly because future access is unknown
- useful as a benchmark

### 3. LRU (Least Recently Used)
Replace the page that has not been used for the longest time.
- based on temporal locality
- usually good
- expensive to implement exactly

### 4. Clock / Second chance
Approximation to LRU using a reference bit.
- practical
- widely used idea
- balances cost and performance

### 5. LFU (Least Frequently Used)
Replace page used least often.
- can work in some workloads
- old frequency history can be misleading

---

## 🟢 Techniques of Storage Management
Relatively simple memory management already involves cooperation among multiple layers:

- **compiler/assembler** define code and data layout
- **linker** resolves references and builds the executable image
- **loader** places the program in memory
- **kernel** manages partitions and sets protection metadata
- **hardware/MMU** translates addresses and enforces bounds and permissions

The process sees a simple memory abstraction, but that abstraction is created jointly by software and hardware.

### 1. No OS support
Programs are loaded and run with little or no memory management help.

**Characteristics**
- simplest model
- often one program at a time
- physical memory addresses are used directly
- no relocation, little protection
- program must be aware of actual memory layout
- little or no isolation between program and system memory

**Pros**
- minimal overhead
- simple implementation
- fast address use because there is little translation logic

**Cons**
- poor flexibility
- unsafe
- limited multiprogramming
- difficult memory sharing
- difficult to move programs in memory
- hard to protect OS memory from user code

**Typical context**
- very early systems
- small embedded systems
- special-purpose machines with fixed software layout

---

### 2. Single-user contiguous allocation
One user process occupies one contiguous region of memory. Usually memory is divided into:
- one part for the OS
- one part for the user program

**Characteristics**
- program must fit in one continuous block
- only one main user process at a time
- user program occupies one large contiguous address range
- execution is simple because the memory layout is predictable

**Pros**
- simple
- easy address generation
- low hardware and OS complexity

**Cons**
- poor CPU utilization
- no true multiprogramming
- limited memory use efficiency
- weak flexibility when program size changes
- no effective way to keep several user processes resident at once

---

### 3. Partitioned allocation
Memory is divided into partitions; each partition holds one process.

This technique is an important step toward **multiprogramming**, because several processes may reside in memory at the same time. Each process is placed into its own region, while the operating system occupies its own protected area.

A key idea introduced in the slides is that each process can be given the illusion of a private, linear address space even though multiple processes are actually stored in different areas of main memory.

#### Virtualization in partitioned allocation
- each process views memory as a private, unshared, linear address space
- addresses generated by the program are not treated as final physical addresses
- hardware can translate a process address by adding a **base register** value
- hardware can also check a **bounds/limit register** to ensure the address stays within the partition
- this creates a simple form of relocation and protection

So the process may behave as though its code starts at address 0, while the MMU maps that logical address into the correct physical partition in memory.

#### Role of system components
Partitioned allocation also connects to the software toolchain and runtime system:

- **compiler/assembler** generate code assuming a process address layout
- **linker** combines object modules into a program image
- **loader** places the program into an available partition
- **kernel** chooses the partition and initializes memory-management registers
- **hardware/MMU** performs address translation during execution

This division of responsibility is important:
- software prepares the executable image
- the OS decides where it goes
- hardware enforces the mapping and protection at runtime

#### Typical process abstraction
Even under simple partitioned allocation, a process is often viewed as having:
- **code**
- **data**
- **heap**
- **stack**

These regions appear conceptually separate to the process, even though in a simple partitioning scheme they may still reside inside one contiguous physical allocation.

#### A. Fixed partitioning
- memory split into fixed-size partitions
- each partition holds one process
- partition count limits the number of concurrent processes
- OS may keep one partition or region for itself and assign the rest to user jobs

**Pros**
- simple
- easy bookkeeping
- straightforward scheduling of resident processes
- simple hardware support when combined with base and bounds registers

**Cons**
- **internal fragmentation**: unused space inside a partition
- process size limited by partition size
- number of concurrent processes limited by partition count
- poor fit for workloads with very different process sizes

#### B. Dynamic partitioning
- partitions created to fit process sizes dynamically
- partition size is based on the actual memory needs of the process at load time
- process may be relocated into any sufficiently large free block

**Pros**
- better memory use than fixed partitions
- process gets a more appropriate block size
- supports more flexible admission of differently sized programs

**Cons**
- **external fragmentation**: free memory broken into many small holes
- may need compaction to combine holes
- placement strategy becomes important
- memory-management bookkeeping becomes more complex

#### Protection in partitioned allocation
The later slides add another major idea: **permission and protection metadata** can be associated with regions such as:
- code
- data
- stack

For each region, hardware may track:
- **base**
- **bounds**
- **permission**

Examples:
- code may be **read/execute**
- data may be **read/write**
- stack may be **read/write**, but not executable

This helps prevent:
- a process from reading or writing outside its assigned partition
- accidental overwriting of code
- direct access into operating system memory
- invalid stack or data accesses beyond legal bounds

So partitioned allocation is not only about placement; it is also an early and important mechanism for **memory protection**.

---

### 4. Segmentation
A program is divided into logical units called segments. Segmentation is essentially **logical organization**.

**Examples**
- code segment
- data segment
- stack segment
- heap segment

Each segment:
- has its own base and length
- can grow/shrink independently
- reflects program structure
- may have its own permissions and sharing rules

This idea is hinted at in the partitioned-allocation slides, where code, data, and stack are shown as distinct conceptual regions with separate base/bounds information. Segmentation generalizes that idea into a full memory-management scheme.

**Advantages**
- matches programmer’s logical view
- supports protection per segment
- supports sharing, such as shared code segments
- no internal fragmentation within variable-sized segments
- supports different access permissions for code, data, and stack

**Disadvantages**
- causes external fragmentation
- more complex allocation
- segment lookup overhead
- segment growth and movement can complicate management

---

### 5. Paging
Paging is physical management by fixed-size units. Memory is divided into fixed-size blocks:
- **pages** in virtual memory
- **frames** in physical memory
- a page can be placed in any free frame

Unlike partitioned allocation, paging does not require one large contiguous region per process. A process can be split across many page frames.

**Advantages**
- eliminates external fragmentation
- easy noncontiguous allocation
- supports virtual memory naturally
- simple replacement unit
- easier to manage many processes simultaneously

**Disadvantages**
- internal fragmentation in the last page
- address translation overhead
- requires page tables
- may cause many page faults if memory is poorly managed

---

### 6. Segmentation and paging
Segmentation provides logical organization, while paging provides efficient physical allocation. This combines both ideas:
- segment the program logically
- page each segment physically

Therefore:
- programmer or compiler sees segments
- OS manages actual memory using pages within segments
- hardware performs multi-step translation and protection checks

This combined approach keeps the logical structure of segmentation while reducing the allocation problems of pure segmentation.

**Advantages**
- keeps logical program structure
- reduces external fragmentation compared with pure segmentation
- supports protection and sharing at segment level
- supports efficient allocation at page level
- allows different regions such as code, data, and stack to retain distinct meaning and permissions

**Disadvantages**
- more complex hardware/software support
- more translation steps
- larger management overhead
- more metadata to maintain

---

## 🟢 Criteria for Memory Management Technique Comparison

### Protection
Protection means preventing one process from illegally accessing:
- another process’s memory
- OS memory
- restricted regions within its own address space

**Without protection:**
- bugs corrupt memory
- malicious code can attack other programs
- OS stability breaks down

**In different techniques**
- No OS support / simple contiguous allocation: weak protection
- Partitioning: some protection via bounds checks
- Segmentation: strong logical protection per segment
- Paging: strong protection per page using access bits
- Segmentation + paging: very flexible protection

Typical protection checks:
- read/write/execute permissions
- valid/invalid bits
- bounds checking
- user vs kernel mode separation

---

### Multiprogramming
Multiprogramming means keeping multiple processes in memory so the CPU can switch among them. Therefore when one process waits for I/O, another can run.

**Technique comparison**
- No OS support: almost none
- Single contiguous allocation: poor
- Partitioned allocation: supports several processes
- Segmentation / paging: strong support
- Virtual memory: lets processes exceed physical RAM size, improving concurrency

**Core point**
Better storage management usually means better degree of multiprogramming.

---

### Level of support
This refers to whether the scheme requires:
- hardware support
- OS support
- compiler/programmer support

**Examples**
- No OS support: little support required
- Contiguous allocation: basic OS help
- Partitioning: OS bookkeeping
- Segmentation: hardware registers/tables + OS management
- Paging: hardware MMU, page tables, TLB, OS page-fault handling
- Segmentation + paging: heavy hardware + OS support

**Important hardware concepts**
- MMU: Memory Management Unit
- TLB: Translation Lookaside Buffer
- base and limit registers
- page table / segment table

---

### Efficiency
Efficiency includes:
- memory utilization
- CPU overhead
- I/O overhead
- translation overhead
- fragmentation behavior

**Fragmentation**
Two major types:
- Internal fragmentation
- Allocated block is bigger than needed.

**Common in:**
- fixed partitions
- paging (especially final page)

**External fragmentation**
- Free memory exists, but scattered in small holes.

**Common in:**
- dynamic partitions
- segmentation

---

### Technique Comparison Overview
**Fixed partitioning:** simple but wasteful internally
**Dynamic partitioning:** better fit, but external fragmentation
**Paging:** good for allocation, low external fragmentation
**Segmentation:** good logical structure, but external fragmentation
**Segmentation + paging:** better balance, more overhead

---

### Relationship to scheduling
Memory management and CPU scheduling are closely related. Therefore a process cannot run unless its needed code/data are in memory.

**Main connections**
1. Degree of multiprogramming affects CPU scheduling - more processes in memory:
- more ready processes
- better CPU utilization
- better scheduling choices

2. Swapping interacts with scheduling - if a process is swapped out:
- scheduler cannot run it immediately
- swap time delays execution

3. Page faults affect response time - a scheduled process may block on a page fault. Then:
- it leaves the CPU
- scheduler picks another process

4. Thrashing destroys scheduling effectiveness - if many processes spend most time paging:
- CPU utilization drops
- scheduler has little useful work to dispatch

5. Working set and resident set affect performance - if a process has enough pages resident:
- it runs smoothly
If not:
- frequent faults occur
- scheduler sees repeated blocking

Scheduling decides which ready process runs next.  |
Storage management helps determine which processes are actually ready and efficient to run.

---

## 🟢 C Code to Program
C source → compiler → assembler → object file → linker → executable → loader → memory

### Compiler
The compiler translates C code into:
- assembly code, or
- directly into object code

It converts high-level constructs like:
- loops
- variables
- function calls
- arrays

into low-level operations like:
- loads/stores
- arithmetic
- branches/jumps

Example:
```
x[i] = y[i];
```
becomes something like:
- compute address of `y[i]`
- load byte
- compute address of `x[i]`
- store byte
---
### Assembler
The assembler translates assembly into machine instructions and produces an object file.

The object file usually contains:
- machine code
- symbol table
- relocation information

**Symbol table**
This records names and where they are defined, such as:
- strcpy
- L1
- L2

**Relocation information**
- This records places where addresses are not final yet and must be fixed later.
---
### Linker
The linker:
- combines object files
- resolves symbol references across files
- selects needed routines from libraries
- assigns addresses within the executable image
- patches instructions/data using relocation info

So if one module calls strcpy, the linker finds where strcpy is and fixes the reference.

The result is an executable (or shared object).

---
### Loader
The loader, working with the OS/kernel, creates a process address space and puts the program there.

Typical regions are:
- text segment: machine code
- data segment: initialized globals/statics
- BSS: uninitialized globals/statics
- heap: dynamic allocation
- stack: function calls, locals, return addresses

So object code ends up in the process’s code/text region.

On modern systems, the program is usually loaded into virtual memory, not directly by the programmer into a fixed physical address.
- linker prepares the executable layout
- loader maps program sections into the process’s virtual address space
- hardware + OS map those virtual pages to physical memory
---
### Sharing code between processes
Read-only machine code can be shared. Machine code in the text segment is usually:
- read-only
- reentrant / safe to execute concurrently

Because it is not modified, the OS can map the same physical code pages into multiple processes.

**Example**
If 20 processes run the same program:
- each process has its own stack and private data
- but the code pages can often be shared

That saves memory.

---
### Shared libraries
- many processes may use the same library code
- only one copy of those code pages may need to reside in physical memory
- each process maps them into its own virtual address space
---
### Private resources
- stack
- heap
- writable globals

Because one process changing them should not affect another.

Some writable pages may begin shared and later become private using copy-on-write.

---